# Delta Pipeline Notebook

Pipeline: `dbt -> Trino -> S3(MinIO) -> read`

Only Delta format is used in this project.


## 1) Helpers

Utility helpers to run Docker/Trino commands and print outputs.


In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import trino

def detect_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Cannot locate repo root (.git not found)')

ROOT = detect_repo_root(Path.cwd().resolve())
COMPOSE_FILE = ROOT / 'infra' / 'docker-compose.trino.yml'
DBT_DIR = ROOT / 'dbt' / 'trino_pipeline'

def run_cmd(cmd: list[str], check: bool = True, cwd: Path | None = None) -> str:
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(cwd) if cwd else None)
    if check and proc.returncode != 0:
        raise RuntimeError("Command failed (%s): %s\n%s" % (proc.returncode, ' '.join(cmd), proc.stderr))
    return proc.stdout.strip()

def trino_sql(sql: str) -> str:
    conn = trino.dbapi.connect(
        host='localhost',
        port=8080,
        user='dbt',
        catalog='delta',
        schema='analytics',
    )
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description] if cur.description else []
    lines = []
    if cols:
        lines.append(' | '.join(cols))
    lines.extend(' | '.join(str(x) for x in r) for r in rows)
    return '\n'.join(lines)


## 2) Run Pipeline

Runs `dbt seed` and `dbt run` in containers.


In [ ]:
print(run_cmd(['make', 'pipeline-run'], cwd=ROOT))


## 3) Delta Catalog Checks

Checks catalogs, schemas, and transformed table output.


In [ ]:
print('Catalogs:')
print(trino_sql('show catalogs'))
print('\nSchemas in delta:')
print(trino_sql('show schemas from delta'))
print('\nRaw dataset row count:')
print(trino_sql('select count(*) as rows_total from delta.raw.titanic_dataset'))
print('\nRaw dataset preview (delta.raw.titanic_dataset):')
print(trino_sql('select PassengerId, Survived, Pclass, Name, Sex, Age from delta.raw.titanic_dataset order by PassengerId limit 10'))
print('\nTransformed table preview:')
print(trino_sql('select * from delta.analytics.survival_by_class order by pclass'))
